# 📈 Modelado - Regresión y Clasificación

## 🎯 1. Objetivo del notebook

En este notebook se entrenan y evalúan los modelos de Machine Learning solicitados para el proyecto.

A partir del dataset limpio generado en el notebook de preprocesamiento, se trabajará con dos objetivos diferentes:

- Un modelo de regresión lineal para predecir la variable `nota_final`.
- Un modelo de regresión logística para predecir la variable `aprobado`.

El objetivo es preparar los datos de forma adecuada, entrenar ambos modelos y evaluar su rendimiento utilizando métricas apropiadas para cada tipo de problema.

En el caso de la regresión, se analizarán métricas como MAE, RMSE y R².  
En el caso de la clasificación, se tendrá en cuenta el desequilibrio observado en la variable `aprobado`, por lo que no se evaluará el modelo únicamente con accuracy.

## 2. Importación de librerías 

In [1]:
# Importación de librerías principales
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

# Herramientas de preparación de datos
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# Modelos
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression

# Métricas para regresión
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Métricas para clasificación
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Configuración de visualización
pd.set_option("display.max_columns", None)

### Importación de librerías

Se importan las librerías necesarias para la fase de modelado.

En este notebook se utilizan herramientas de `scikit-learn` para dividir los datos en entrenamiento y prueba, aplicar transformaciones a las variables, crear pipelines y entrenar los modelos de regresión lineal y regresión logística.

También se importan métricas específicas para evaluar cada tipo de problema: métricas de error para regresión y métricas de clasificación para el modelo logístico.

## 3. Carga de datos

In [2]:
# Definición de la ruta del dataset procesado
ruta_processed = Path("../Data/processed/dataset_estudiantes_limpio.csv")

# Carga del dataset limpio
df_model = pd.read_csv(ruta_processed)

# Mostramos las primeras filas
df_model.head()


,horas_estudio_semanal,nota_anterior,tasa_asistencia,horas_sueno,edad,nivel_dificultad,tiene_tutor,horario_estudio_preferido,estilo_aprendizaje,nota_final,aprobado
0,8.957476,48.830601,86.640182,6.675694,25,Fácil,Sí,Tarde,Lectura/Escritura,84.4,1
1,11.042524,80.825707,83.449655,4.616844,18,Difícil,No,Tarde,Desconocido,72.0,1
2,4.510776,90.383694,74.623607,7.755246,25,Fácil,No,Mañana,Lectura/Escritura,80.0,1
3,6.647213,81.878257,82.849841,8.592826,23,Fácil,No,Desconocido,Visual,78.2,1
4,1.000000,66.254179,54.539935,6.671840,21,Medio,No,Desconocido,Auditivo,66.0,1


### Carga del dataset procesado

Se carga el dataset limpio generado en el notebook de preprocesamiento.

A diferencia del análisis exploratorio, en esta fase ya se trabaja con una versión sin valores nulos y preparada para continuar con el modelado.

El objetivo es utilizar este dataset como base para entrenar los modelos de regresión y clasificación.

## 4. Revisión del dataset

Aunque ya lo revisamos antes, aquí hacemos una comprobación rápida para asegurarnos de que todo está correcto antes de preparar los modelos.

### 4.1 Dimensiones del dataset

In [3]:
# Dimensiones del dataset procesado
print(f"El dataset procesado contiene {df_model.shape[0]} filas y {df_model.shape[1]} columnas.")


El dataset procesado contiene 1000 filas y 11 columnas.


### 4.2 Información general

In [4]:
# Información general del dataset procesado
df_model.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   horas_estudio_semanal      1000 non-null   float64
 1   nota_anterior              1000 non-null   float64
 2   tasa_asistencia            1000 non-null   float64
 3   horas_sueno                1000 non-null   float64
 4   edad                       1000 non-null   int64  
 5   nivel_dificultad           1000 non-null   object 
 6   tiene_tutor                1000 non-null   object 
 7   horario_estudio_preferido  1000 non-null   object 
 8   estilo_aprendizaje         1000 non-null   object 
 9   nota_final                 1000 non-null   float64
 10  aprobado                   1000 non-null   int64  
dtypes: float64(5), int64(2), object(4)
memory usage: 86.1+ KB


### 4.3 Comprobación de valores nulos y duplicados

In [5]:
# Comprobación rápida de valores nulos y duplicados
print(f"Valores nulos totales: {df_model.isnull().sum().sum()}")
print(f"Registros duplicados: {df_model.duplicated().sum()}")

Valores nulos totales: 0
Registros duplicados: 0


### Revisión del dataset

Antes de separar las variables predictoras y las variables objetivo, se realiza una comprobación rápida del dataset procesado.

Se revisan las dimensiones, los tipos de datos, la existencia de valores nulos y la presencia de duplicados.

Esta comprobación ayuda a confirmar que el archivo generado en el preprocesamiento se ha cargado correctamente y que está listo para iniciar la fase de modelado.

## 5. Separación de variables predictoras y variables objetivo

En esta fase se separan las variables que se utilizarán como entrada de los modelos y las variables que se quieren predecir.

Para el problema de regresión, la variable objetivo será `nota_final`.

Para el problema de clasificación, la variable objetivo será `aprobado`.

Es importante no utilizar `aprobado` como variable predictora para predecir `nota_final`, ya que se deriva de la propia nota final. Del mismo modo, tampoco se debe utilizar `nota_final` para predecir `aprobado`, porque esto provocaría fuga de información y el modelo tendría acceso directo a la respuesta.

### 5.1 Definir variables predictoras

In [6]:
# Definición de variables predictoras comunes para ambos modelos
variables_predictoras = [
    "horas_estudio_semanal",
    "nota_anterior",
    "tasa_asistencia",
    "horas_sueno",
    "edad",
    "nivel_dificultad",
    "tiene_tutor",
    "horario_estudio_preferido",
    "estilo_aprendizaje"
]

# Mostramos las variables seleccionadas
variables_predictoras

['horas_estudio_semanal',
 'nota_anterior',
 'tasa_asistencia',
 'horas_sueno',
 'edad',
 'nivel_dificultad',
 'tiene_tutor',
 'horario_estudio_preferido',
 'estilo_aprendizaje']

### 5.2 Definir variables objetivo

In [7]:
# Definición de variables objetivo
objetivo_regresion = "nota_final"
objetivo_clasificacion = "aprobado"

print(f"Variable objetivo para regresión: {objetivo_regresion}")
print(f"Variable objetivo para clasificación: {objetivo_clasificacion}")

Variable objetivo para regresión: nota_final
Variable objetivo para clasificación: aprobado


### 5.3 Creación 'X' e 'y' para regresión

In [8]:
# Separación de variables para el modelo de regresión
X_reg = df_model[variables_predictoras]
y_reg = df_model[objetivo_regresion]

print("Dimensiones de X_reg:", X_reg.shape)
print("Dimensiones de y_reg:", y_reg.shape)

Dimensiones de X_reg: (1000, 9)
Dimensiones de y_reg: (1000,)


### 5.4 Creación 'X' e 'y' para clasificación

In [9]:
# Separación de variables para el modelo de clasificación
X_clf = df_model[variables_predictoras]
y_clf = df_model[objetivo_clasificacion]

print("Dimensiones de X_clf:", X_clf.shape)
print("Dimensiones de y_clf:", y_clf.shape)

Dimensiones de X_clf: (1000, 9)
Dimensiones de y_clf: (1000,)


### Variables utilizadas en los modelos

Se utilizan las mismas variables predictoras para los dos modelos, ya que todas ellas representan información disponible antes de conocer el resultado final del estudiante.

Para el modelo de regresión, el objetivo será predecir la nota final obtenida por el estudiante.

Para el modelo de clasificación, el objetivo será predecir si el estudiante aprueba o no.

Se excluyen tanto `nota_final` como `aprobado` del conjunto de variables predictoras para evitar problemas de fuga de información.

Las matrices `X_reg` y `X_clf` contienen las variables predictoras utilizadas por los modelos. En ambos casos tienen 1000 registros y 9 variables de entrada.

Las variables `y_reg` y `y_clf` contienen únicamente la variable objetivo de cada problema. Por eso aparecen con dimensión `(1000,)`, ya que son series de pandas con un único valor objetivo por cada registro del dataset.

### 5.5 Separar variables numéricas y categóricas

In [10]:
# Identificación de variables numéricas y categóricas dentro de las predictoras
variables_numericas = X_reg.select_dtypes(include=["int64", "float64"]).columns.tolist()
variables_categoricas = X_reg.select_dtypes(include=["object"]).columns.tolist()

print("Variables numéricas:")
print(variables_numericas)

print("\nVariables categóricas:")
print(variables_categoricas)

Variables numéricas:
['horas_estudio_semanal', 'nota_anterior', 'tasa_asistencia', 'horas_sueno', 'edad']

Variables categóricas:
['nivel_dificultad', 'tiene_tutor', 'horario_estudio_preferido', 'estilo_aprendizaje']


### Identificación de variables numéricas y categóricas

Antes de entrenar los modelos, se separan las variables predictoras en numéricas y categóricas.

Esta separación es necesaria porque cada tipo de variable requiere un tratamiento diferente.

Las variables numéricas podrán escalarse, mientras que las variables categóricas deberán transformarse mediante One-Hot Encoding para que puedan ser interpretadas por los modelos.

## 6. División en conjuntos de entrenamiento y prueba

Antes de entrenar los modelos, se divide el dataset en un conjunto de entrenamiento y un conjunto de prueba.

El conjunto de entrenamiento se utilizará para ajustar los modelos, mientras que el conjunto de prueba servirá para evaluar su rendimiento con datos no utilizados durante el aprendizaje.

Se realiza una división independiente para el problema de regresión y para el problema de clasificación. En el caso de la clasificación, se utiliza distribución en capas para mantener la proporción original de estudiantes aprobados y no aprobados en ambos conjuntos.


### 6.1 División train-test para regresión

In [11]:
# División en entrenamiento y prueba para el modelo de regresión
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.2,
    random_state=42
)

# Comprobamos las dimensiones resultantes
print("X_train_reg:", X_train_reg.shape)
print("X_test_reg:", X_test_reg.shape)
print("y_train_reg:", y_train_reg.shape)
print("y_test_reg:", y_test_reg.shape)

X_train_reg: (800, 9)
X_test_reg: (200, 9)
y_train_reg: (800,)
y_test_reg: (200,)


### División para el modelo de regresión

Para el modelo de regresión se divide el dataset utilizando un 80% de los datos para entrenamiento y un 20% para prueba.

La variable objetivo `nota_final` es continua, por lo que no se aplica estratificación en esta división.

Se utiliza `random_state=42` para que la división sea reproducible y se obtengan los mismos resultados al volver a ejecutar el notebook.

### 6.2 División train-test para clasificación

In [12]:
# División en entrenamiento y prueba para el modelo de clasificación
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf,
    y_clf,
    test_size=0.2,
    random_state=42,
    stratify=y_clf
)

# Comprobamos las dimensiones resultantes
print("X_train_clf:", X_train_clf.shape)
print("X_test_clf:", X_test_clf.shape)
print("y_train_clf:", y_train_clf.shape)
print("y_test_clf:", y_test_clf.shape)

X_train_clf: (800, 9)
X_test_clf: (200, 9)
y_train_clf: (800,)
y_test_clf: (200,)


### División para el modelo de clasificación

Para el modelo de clasificación también se utiliza una división 80/20 entre entrenamiento y prueba.

En este caso sí se aplica `stratify=y_clf`, ya que la variable `aprobado` presenta un desequilibrio claro entre clases.

La estratificación permite mantener una proporción similar de aprobados y no aprobados tanto en el conjunto de entrenamiento como en el conjunto de prueba. Esto es importante para que la evaluación del modelo sea más representativa.

### 6.3 Comprobación de la distribución de clases después de la división

In [13]:
# Comprobamos la distribución de clases en el conjunto original, entrenamiento y prueba
distribucion_clases = pd.DataFrame({
    "dataset_completo": y_clf.value_counts(normalize=True).mul(100).round(2),
    "entrenamiento": y_train_clf.value_counts(normalize=True).mul(100).round(2),
    "prueba": y_test_clf.value_counts(normalize=True).mul(100).round(2)
})

distribucion_clases

,dataset_completo,entrenamiento,prueba
aprobado,,,
1,89.8,89.75,90.0
0,10.2,10.25,10.0


### Comprobación de la distribución de clases

Después de dividir los datos para el modelo de clasificación, se revisa la distribución de la variable `aprobado` en el dataset completo, en el conjunto de entrenamiento y en el conjunto de prueba.

Esta comprobación permite verificar que la proporción de estudiantes aprobados y no aprobados se mantiene de forma similar en los diferentes conjuntos.

Mantener esta proporción es importante porque el dataset original está desequilibrado y una división aleatoria sin estratificación podría generar conjuntos poco representativos.

### 6.4 Conclusiones de la división train-test

Se han creado correctamente los conjuntos de entrenamiento y prueba para los dos problemas del proyecto.

Para el modelo de regresión, se ha realizado una división estándar 80/20, reservando el 20% de los datos para evaluar el rendimiento del modelo sobre registros no utilizados durante el entrenamiento.

Para el modelo de clasificación, se ha aplicado estratificación sobre la variable `aprobado`, ya que en el EDA se observó un claro desequilibrio entre clases.

La comprobación posterior muestra que la proporción de estudiantes aprobados y no aprobados se mantiene prácticamente igual en el dataset completo, en el conjunto de entrenamiento y en el conjunto de prueba. Esto permite que la evaluación del modelo de clasificación sea más representativa.

## 7. Preparación de transformaciones para los modelos

Antes de entrenar los modelos, es necesario preparar las variables predictoras para que puedan ser utilizadas correctamente por los algoritmos.

Las variables numéricas se escalarán mediante `StandardScaler`, ya que tienen rangos diferentes y esto puede ayudar especialmente al modelo de regresión logística.

Las variables categóricas se transformarán mediante `OneHotEncoder`, convirtiendo cada categoría en variables numéricas binarias.

Estas transformaciones se aplicarán dentro de un `ColumnTransformer`, lo que permite tratar de forma diferente las variables numéricas y categóricas dentro de un mismo proceso.

### 7.1 Definir transformaciones para columnas numéricas y categóricas

In [14]:
# Transformación para variables numéricas
transformador_numerico = StandardScaler()

# Transformación para variables categóricas
transformador_categorico = OneHotEncoder(
    drop="first",
    handle_unknown="ignore"
)

### Transformaciones seleccionadas

Para las variables numéricas se utiliza `StandardScaler`, que transforma los valores para que tengan media 0 y desviación estándar 1.

Para las variables categóricas se utiliza `OneHotEncoder`. Además, se aplica `drop="first"` para evitar crear columnas redundantes, algo especialmente recomendable en modelos lineales.

También se utiliza `handle_unknown="ignore"` para evitar errores si en el conjunto de prueba aparece alguna categoría no vista durante el entrenamiento.

### 7.2 Crear el preprocesador

In [16]:
# Creación del preprocesador general
preprocesador = ColumnTransformer(
    transformers=[
        ("num", transformador_numerico, variables_numericas),
        ("cat", transformador_categorico, variables_categoricas)
    ]
)

preprocesador

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

### Creación del preprocesador

Se crea un `ColumnTransformer` para aplicar transformaciones diferentes según el tipo de variable.

A las variables numéricas se les aplicará escalado, mientras que a las variables categóricas se les aplicará codificación One-Hot.

Este preprocesador se incorporará posteriormente dentro de los pipelines de regresión y clasificación, de forma que las transformaciones se ajusten únicamente con los datos de entrenamiento.


### 7.3 Comprobar las variables que entran en cada transformación

In [17]:
# Comprobación de variables incluidas en cada transformación
print("Variables numéricas que se escalarán:")
print(variables_numericas)

print("\nVariables categóricas que se codificarán:")
print(variables_categoricas)

Variables numéricas que se escalarán:
['horas_estudio_semanal', 'nota_anterior', 'tasa_asistencia', 'horas_sueno', 'edad']

Variables categóricas que se codificarán:
['nivel_dificultad', 'tiene_tutor', 'horario_estudio_preferido', 'estilo_aprendizaje']


### Comprobación de variables transformadas

Se revisan las variables que se incluirán en cada transformación antes de construir los modelos.

Esta comprobación ayuda a verificar que todas las variables predictoras se han clasificado correctamente como numéricas o categóricas.

### 7.4 Comprobación sobre el conjunto de entrenamiento

In [18]:
# Comprobación del funcionamiento del preprocesador sobre el conjunto de entrenamiento
X_train_reg_preprocesado = preprocesador.fit_transform(X_train_reg)

# Dimensiones después de aplicar las transformaciones
print("Dimensiones originales de X_train_reg:", X_train_reg.shape)
print("Dimensiones después del preprocesamiento:", X_train_reg_preprocesado.shape)

Dimensiones originales de X_train_reg: (800, 9)
Dimensiones después del preprocesamiento: (800, 15)


### Comprobación del preprocesador

Se aplica el preprocesador sobre el conjunto de entrenamiento de regresión para comprobar que las transformaciones funcionan correctamente.

Después del preprocesamiento, el número de columnas aumenta porque las variables categóricas se convierten en varias columnas binarias mediante One-Hot Encoding.

Esta comprobación sirve únicamente para validar el proceso. En los modelos finales, el preprocesamiento se aplicará dentro de pipelines para evitar fugas de información.

### 7.5 Obtención de los nombres de las columnas generadas

In [19]:
# Obtención de nombres de variables transformadas
nombres_variables_transformadas = preprocesador.get_feature_names_out()

# Mostramos los nombres generados
nombres_variables_transformadas


array(['num__horas_estudio_semanal', 'num__nota_anterior',
       'num__tasa_asistencia', 'num__horas_sueno', 'num__edad',
       'cat__nivel_dificultad_Fácil', 'cat__nivel_dificultad_Medio',
       'cat__tiene_tutor_Sí', 'cat__horario_estudio_preferido_Mañana',
       'cat__horario_estudio_preferido_Noche',
       'cat__horario_estudio_preferido_Tarde',
       'cat__estilo_aprendizaje_Desconocido',
       'cat__estilo_aprendizaje_Kinestésico',
       'cat__estilo_aprendizaje_Lectura/Escritura',
       'cat__estilo_aprendizaje_Visual'], dtype=object)

### Variables generadas tras el preprocesamiento

Después de aplicar el preprocesador, se obtienen los nombres de las variables resultantes.

Las variables numéricas mantienen su información original, aunque escalada, mientras que las variables categóricas se transforman en nuevas columnas binarias.

Esta revisión ayuda a entender cómo se representarán los datos antes de entrar en los modelos.

### 7.6 Conclusiones de la preparación de transformaciones

Se ha definido un preprocesador que aplica escalado a las variables numéricas y codificación One-Hot a las variables categóricas.

La comprobación realizada sobre el conjunto de entrenamiento confirma que las transformaciones funcionan correctamente y que las variables categóricas se convierten en variables numéricas aptas para los modelos.

En los siguientes apartados, este preprocesador se integrará dentro de pipelines para entrenar el modelo de regresión lineal y el modelo de regresión logística de forma ordenada y evitando fuga de información.